# **FOREWORD**

This is a simple blend kernel that accepts 3 public notebooks as starter kernels - 

1. [LB 0.825](https://www.kaggle.com/code/kdmitrie/birdclef26-google-perch-starter)
2. [LB 0.792](https://www.kaggle.com/code/waterjoe/birdclef2026-submit-baseline)

I also use the same fallback folder to be able to visualize the blend properly, this has no effect on the actual test set though.

All credits to the base work. Best regards!

**My other kernels in the assignment** <br>
1. [Preprocessor](https://www.kaggle.com/code/ravi20076/birdclef2026-preprocessing-v1)
2. [Baseline OpenVino inference](https://www.kaggle.com/code/ravi20076/birdclef2026-openvino-starter-v1)
3. [Supplements](https://www.kaggle.com/code/ravi20076/birdclef2026-supplements-v1)
4. [Blend-V1](https://www.kaggle.com/code/ravi20076/birdclef2026-public-blend-v1)

# **SCRIPT LIBRARY**

These are the individual scrits used for the blend. One may add more scripts to this kernel here / link this with a GitHub repo as well.

## **LB 0.825**

In [1]:
%%writefile lb825.py

import time
START = time.time()

from concurrent.futures import ThreadPoolExecutor
import glob
import librosa
import numpy as np
import os
import pandas as pd
import re
import sys
import tensorflow as tf

tf.experimental.numpy.experimental_enable_numpy_behavior()

TERMINATE_TIME  = START + 4800
SUBMISSION_FILE = "lb825.csv"

# Import the model
birdclassifier = tf.saved_model.load(
    '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/'
)

bc_labels = pd.read_csv(
    '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/assets/labels.csv'
)

bc_labels = bc_labels.reset_index().rename(
    {'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1
).set_index('scientific_name')

taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
mapping  = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping.bc_index = mapping.bc_index.fillna(len(bc_labels)).astype(int)
mapping  = mapping[['primary_label', 'bc_index']].set_index('primary_label')

primary_labels = pd.read_csv(
    '/kaggle/input/competitions/birdclef-2026/sample_submission.csv'
).columns[1:].to_list()
birdclassifier_indices = [int(mapping.loc[pl][0]) for pl in primary_labels]
birdclassifier_indices[:5]

total_predicted_species = len(set(birdclassifier_indices) - {len(bc_labels)})
print(f'Note: we can predict {total_predicted_species} out of 234 species only!')

def get_oggs(max_oggs=8):
    if len(glob.glob('/kaggle/input/competitions/birdclef-2026/test_soundscapes/*.ogg')) > 0:
        oggs = glob.glob('/kaggle/input/competitions/birdclef-2026/test_soundscapes/*.ogg')
    else:
        oggs = sorted(glob.glob(
            f'/kaggle/input/competitions/birdclef-2026/train_soundscapes/*.ogg')
        )[:max_oggs]
    return [(n, ogg, re.search(r'/([^/]+)\.ogg$', ogg).group(1), n < int(len(oggs))) for n, ogg in enumerate(oggs)]

oggs = get_oggs()

def perch_result(ogg):
    _, fname, ss_id, do_prediction = ogg
    sr = 32_000

    print(f'{ss_id}')
    row_ids = [f'{ss_id}_{n}' for n in range(5, 65, 5)]

    if not do_prediction or time.time() > TERMINATE_TIME:
        return row_ids, -1000 * np.ones((12, len(primary_labels)))
        
    data, _ = librosa.load(fname, sr=sr)
    model_outputs = birdclassifier.signatures['serving_default'](inputs=data.reshape((-1, 5 * sr)))['label']

    model_outputs = tf.pad(model_outputs, tf.constant([[0, 0], [0, 1]]))
    result = model_outputs[:, birdclassifier_indices]

    return row_ids, result

row_ids = []
result = []

with ThreadPoolExecutor(max_workers=4) as executor:
    for ogg_row_ids, ogg_result in executor.map(perch_result, oggs):
        row_ids += ogg_row_ids
        result.append(ogg_result)

submission = pd.DataFrame(np.concatenate(result), columns=primary_labels)
submission['row_id'] = row_ids
submission = submission[['row_id'] + primary_labels]

submission.to_csv(SUBMISSION_FILE, index=False)
print(f"Submission saved: {submission.shape}\n\n")


Writing lb825.py


## **LB 0.792**

In [2]:
%%writefile lb792.py

from warnings import filterwarnings
filterwarnings("ignore")

import os
import glob
import timm
import torch
import torch.nn as nn
import torchaudio
import torchvision
import numpy as np
import pandas as pd
import openvino as ov
from tqdm import tqdm
from torch.utils.data import Dataset

ov_device       = "CPU"
PATH            = '/kaggle/input/competitions/birdclef-2026/'
TEST_PATH       = PATH + 'test_soundscapes/'
TRAIN_PATH      = PATH + 'train_soundscapes/'
N_FALLBACK      = 2
SUBMISSION_FILE = "lb792.csv"
model_dir       = "/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models"

DUR = 5
SR  = 32000

class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512,
                 f_min=20, f_max=16000, channels=1, norm="slaney",
                 mel_scale="htk", target_size=(256, 256), top_db=80.0, **kwargs):
        super().__init__()
        self.channels = channels
        self.top_db   = top_db
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, f_min=f_min, f_max=f_max,
            mel_scale=mel_scale, pad_mode="reflect", power=2.0,
            norm=norm, center=True,
        )
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin     = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S))
        if self.top_db is not None:
            max_val  = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x, resize=True):
        squeeze = x.dim() == 1
        if squeeze:
            x = x.unsqueeze(0)
        mel = self.mel_transform(x)
        mel = self.power_to_db(mel)
        mel = mel.unsqueeze(1).repeat(1, self.channels, 1, 1)
        if resize:
            mel = self.resize(mel)
        B, C = mel.shape[:2]
        flat = mel.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel  = (mel - mins) / (maxs - mins + 1e-7)
        if squeeze:
            mel = mel.squeeze(0)
        return mel

class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        cfg = {
            'backbone':        'tf_efficientnetv2_b0',
            'backbone_pooling':'avg',
            'dropout':          0.1,
            'pretrained':       False,
            'channels':         1,
            'num_labels':       234, # Will dynamically update
        }
        if config:
            cfg.update(config)
        self.backbone = timm.create_model(
            cfg['backbone'],
            pretrained=cfg['pretrained'],
            num_classes=cfg['num_labels'],
            global_pool=cfg['backbone_pooling'],
            in_chans=cfg['channels'],
            drop_rate=cfg['dropout'],
        )

    def forward(self, x):
        return self.backbone(x)


class BirdDataset(Dataset):
    def __init__(self, paths, spec_transform):
        self.paths = paths
        self.spec  = spec_transform
        self.chunk_len = SR * DUR
        self.half_chunk = self.chunk_len // 2

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        filepath = self.paths[idx]
        filename = filepath.split('/')[-1].split('.')[0]
        try:
            wav, _ = torchaudio.load(filepath)
            wav = wav.float()[0]  
            
            n_seg = len(wav) // self.chunk_len 
            
            if n_seg == 0:
                wav = torch.nn.functional.pad(wav, (0, self.chunk_len - len(wav)))
                n_seg = 1
            else:
                wav = wav[: n_seg * self.chunk_len]
            
            # 1. Standard Chunks (0-5s, 5-10s, ...)
            wav_std = wav.reshape((n_seg, self.chunk_len))
            
            # 2. Shifted Chunks for TTA (2.5-7.5s, 7.5-12.5s, ...)
            wav_padded = torch.nn.functional.pad(wav, (0, self.half_chunk))
            wav_shift = torch.stack([
                wav_padded[i * self.chunk_len + self.half_chunk : (i+1) * self.chunk_len + self.half_chunk]
                for i in range(n_seg)
            ])
            
            mel_std   = torch.stack([self.spec(wav_std[i]) for i in range(n_seg)])
            mel_shift = torch.stack([self.spec(wav_shift[i]) for i in range(n_seg)])
            names =[f"{filename}_{int((i + 1) * DUR)}" for i in range(n_seg)]
            
            return mel_std.numpy().astype(np.float32), mel_shift.numpy().astype(np.float32), names
            
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            n_seg = int(240 / DUR) 
            mel_std   = torch.zeros((n_seg, 1, 256, 256))
            mel_shift = torch.zeros((n_seg, 1, 256, 256))
            names =[f"{filename}_{int((i + 1) * DUR)}" for i in range(n_seg)]
            return mel_std.numpy().astype(np.float32), mel_shift.numpy().astype(np.float32), names

def export_to_openvino(ckpt_path, ov_model_path, config=None):
    if os.path.exists(ov_model_path):
        return
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    
    num_labels = state['backbone.classifier.weight'].shape[0]
    cfg = config.copy() if config else {}
    cfg['num_labels'] = num_labels
    
    model = BirdModel(cfg)
    model.load_state_dict(state)
    model.eval()
    
    dummy    = torch.zeros(1, 1, 256, 256)
    ov_model = ov.convert_model(model, example_input=dummy)
    ov_model.reshape([-1, 1, 256, 256]) # Support dynamic batch sizes
    ov.save_model(ov_model, ov_model_path)

def compile_ov_model(ov_model_path):
    core     = ov.Core()
    ov_model = core.read_model(ov_model_path)
    return core.compile_model(ov_model, device_name=ov_device)

def predict_batch(compiled_models, batch_np):
    logits_list =[]
    for compiled in compiled_models:
        logits = compiled(batch_np)[compiled.output(0)]
        logits_list.append(logits)
    
    # 1. Ensemble average the logits directly
    mean_logits = np.mean(logits_list, axis=0)
    
    # 2. Standard sigmoid (removed temperature scaling to preserve native calibration)
    probs = 1.0 / (1.0 + np.exp(np.clip(-mean_logits, -50, 50)))
    return probs

def run_inference(compiled_models, paths, spec):
    dataset  = BirdDataset(paths, spec)
    all_preds, all_names = [],[]

    for mel_std, mel_shift, names in tqdm(dataset, desc="Infer"):
        
        preds_std   = predict_batch(compiled_models, mel_std)
        preds_shift = predict_batch(compiled_models, mel_shift)
        
        # 1. Weave TTA Shifted Predictions (Recovers birds cut in half)
        preds = np.copy(preds_std)
        n = len(preds)
        for i in range(n):
            shift_prev = preds_shift[i-1] if i > 0 else preds_std[i]
            shift_curr = preds_shift[i]
            # 50% Standard Window + 50% overlapping halves
            preds[i] = 0.50 * preds_std[i] + 0.25 * shift_prev + 0.25 * shift_curr
        
        # 2. Wider Gaussian Time-Domain Smoothing
        smoothed_preds = np.copy(preds)
        for i in range(n):
            p_m2 = preds[max(0, i-2)]
            p_m1 = preds[max(0, i-1)]
            p_0  = preds[i]
            p_p1 = preds[min(n-1, i+1)]
            p_p2 = preds[min(n-1, i+2)]
            # Smooth the probabilities out across a 25-second window
            smoothed_preds[i] = 0.05*p_m2 + 0.15*p_m1 + 0.60*p_0 + 0.15*p_p1 + 0.05*p_p2
            
        # 3. Global File Context (The Grandmaster Trick, without clipping!)
        file_max = np.max(smoothed_preds, axis=0)
        
        # Blend: 80% local chunk + 20% global file context
        # This securely boosts the AUC rank of chunks in files where the bird is confirmed to exist
        final_preds = 0.80 * smoothed_preds + 0.20 * file_max
        
        all_preds.append(final_preds)
        all_names.extend(names)

    return np.concatenate(all_preds, axis=0), all_names


def create_submission(preds, names, label_cols):
    sample_sub = pd.read_csv(PATH + 'sample_submission.csv')
    expected_cols = sample_sub.columns.tolist()
    
    if len(preds) == 0:
        sample_sub.to_csv(SUBMISSION_FILE, index=False)
        return sample_sub
        
    df_preds = pd.DataFrame(preds, columns=label_cols[:preds.shape[1]])
    df_preds['row_id'] = names
    
    final_dict = {'row_id': df_preds['row_id']}
    for col in expected_cols:
        if col == 'row_id': continue
        if col in df_preds.columns:
            final_dict[col] = df_preds[col].values
        else:
            final_dict[col] = 0.0
            
    final_df = pd.DataFrame(final_dict)
    final_df = final_df[expected_cols]
    final_df.to_csv(SUBMISSION_FILE, index=False)
    
    print(f"Submission saved: {final_df.shape}\n\n")
    return final_df

def main():
    sample_sub = pd.read_csv(PATH + 'sample_submission.csv')
    label_cols = [c for c in sample_sub.columns if c != 'row_id']

    # 1. Search for test files (Using glob to support recursive search)
    paths = glob.glob(TEST_PATH + "**/*.ogg", recursive=True)
    
    if not paths:
        paths = sorted(glob.glob(TRAIN_PATH + "**/*.ogg", recursive=True))[:N_FALLBACK]

    # 3. Handle absolutely no files case elegantly!
    if not paths:
        print(f"Files: 0. Generating default submission directly to avoid crashes.")
        create_submission([], [], label_cols)
        return

    print(f"Files: {len(paths)}, Model Output Classes: {len(label_cols)}")

    spec = Spectrogram(sr=SR, n_fft=2048, n_mels=256, hop_length=512,
                       f_min=20, f_max=16000, channels=1,
                       target_size=(256, 256), top_db=80.0)

    all_ckpts =[os.path.join(model_dir, f) for f in sorted(os.listdir(model_dir)) if f.endswith('.pth')]
    best_ckpts =[p for p in all_ckpts if '_best.pth' in os.path.basename(p)]
    ckpt_paths = best_ckpts if best_ckpts else all_ckpts

    ov_export_dir = "/kaggle/working/ov_models"
    os.makedirs(ov_export_dir, exist_ok=True)

    compiled_models =[]
    for ckpt in ckpt_paths:
        ov_name = os.path.splitext(os.path.basename(ckpt))[0] + '.xml'
        ov_path = os.path.join(ov_export_dir, ov_name)
        export_to_openvino(ckpt, ov_path)
        compiled_models.append(compile_ov_model(ov_path))

    print(f"Loaded {len(compiled_models)} OpenVINO model(s)")

    preds, names = run_inference(compiled_models, paths, spec)
    create_submission(preds, names, label_cols)

if __name__ == "__main__":
    main()

Writing lb792.py


## **BLEND**

In [3]:
%%writefile blend.py

import pandas as pd, os
from warnings import filterwarnings
filterwarnings("ignore")

print(f"\n---> Starting model blend")

sub825 = pd.read_csv(f"lb825.csv", index_col = "row_id")
sub792 = pd.read_csv(f"lb792.csv", index_col = "row_id")[sub825.columns]

sub_fl = sub792 * 0.25 + sub825 * 0.75

print(f"---> Blended submission file shape = {sub_fl.shape}\n")
sub_fl.to_csv(f"submission.csv", index = True)

Writing blend.py


# **SUBMISSION**

In [4]:
!pip install -q \
    /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl
!pip install -q \
    /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.20.0 which is incompatible.


In [5]:
import pandas as pd, os, time, sys

!python lb825.py
!python lb792.py
!python blend.py

print()
display(
    pd.read_csv(
        "submission.csv", index_col = "row_id"
    ).head(5)
)

print()
!ls

2026-03-12 14:16:01.778899: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/kaggle/working/lb825.py:41: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  birdclassifier_indices = [int(mapping.loc[pl][0]) for pl in primary_labels]
Note: we can predict 203 out of 234 species only!
BC2026_Train_0001_S08_20250606_030007
BC2026_Train_0002_S08_20250607_030007
BC2026_Train_0004_S08_20250607_070007
BC2026_Train_0003_S08_20250607_070007
I0000 00:00:1773324984.569220      84 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
BC2026_Train_0005_S08_20250607_070007
BC2026_Train_0006_S09_20250828_000000
BC2026_Train_0007_S09_20250829_000000
BC2026

,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
row_id,,,,,,,,,,,,,,,,,,,,,
BC2026_Train_0001_S08_20250606_030007_10,0.049130,0.000003,-1.241135,2.487681e-08,0.000277,0.789285,-1.484146,-0.281371,-1.929810,-3.266195,...,-0.168471,2.508234,1.226902,1.006112,-0.833552,1.446368,-0.242112,-0.271963,1.461058,2.057732
BC2026_Train_0001_S08_20250606_030007_15,-0.021050,0.000003,-1.151354,2.687707e-08,0.000319,0.643658,-1.400191,-0.530576,-1.869122,-3.204519,...,-0.068266,2.433632,0.895199,1.018849,-0.528212,2.066083,-0.216168,-0.112997,1.281820,1.402287
BC2026_Train_0001_S08_20250606_030007_20,0.214826,0.000003,-0.952940,3.295504e-08,0.000301,0.815393,-1.305014,-0.287078,-1.821816,-3.397747,...,0.168998,2.526814,0.443089,0.567365,-0.335995,1.992599,-0.191512,0.060394,1.355216,1.662714
BC2026_Train_0001_S08_20250606_030007_25,0.389931,0.000002,-1.178950,2.511125e-08,0.000256,0.627937,-1.420967,-0.331734,-1.756993,-3.261578,...,-0.365925,2.620754,0.652351,0.608081,-0.793554,1.995350,-0.242599,-0.102313,1.504851,2.298766
BC2026_Train_0001_S08_20250606_030007_30,-0.035666,0.000002,-1.026649,2.482991e-08,0.000224,0.631065,-1.285451,-0.378177,-1.617058,-3.274080,...,-0.454172,3.164075,1.040866,0.978826,-0.042009,2.765890,-0.139197,-0.454815,1.914434,2.339486



blend.py   lb792.py   lb825.py		  ov_models
lb792.csv  lb825.csv  __notebook__.ipynb  submission.csv
